In [ ]:
import math as m
from glob import glob
from os.path import basename, join
import warnings
from itertools import combinations
import numpy as np
import pandas as pd
import scipy as sp
import cv2
import skimage as si
from tqdm.auto import tqdm
import plotly.express as px
from ANONYMIZED_code.tools.utils import VideoFrames, rescale_to_height

In [ ]:
video_paths = sorted(glob("data/ANONYMIZED/with_tools/*.mov"))
tool_mask_video_paths = [
    p.replace("/with_tools/", "/video_masks/").removesuffix(".mov") + ".mp4"
    for p in video_paths
]

In [ ]:
step_ms = 50
rescale_target_height = 128

In [ ]:
def image_arrs_tool_mask_arrs2mse_hausdorff(
    image_arr_1, image_arr_2, tool_mask_arr_1, tool_mask_arr_2
):
    image_arr_1 = si.util.img_as_float32(image_arr_1)
    image_arr_2 = si.util.img_as_float32(image_arr_2)
    tool_mask_arr_1 = si.util.img_as_ubyte(tool_mask_arr_1).max(axis=2) > 100
    tool_mask_arr_2 = si.util.img_as_ubyte(tool_mask_arr_2).max(axis=2) > 100
    tool_in_either = tool_mask_arr_1 | tool_mask_arr_2
    no_tool_in_either = ~tool_in_either
    mse = (
        (image_arr_1[no_tool_in_either] - image_arr_2[no_tool_in_either]) ** 2
    ).mean()
    tool_points_1 = np.array(np.where(tool_mask_arr_1)).T
    tool_points_2 = np.array(np.where(tool_mask_arr_2)).T
    hausdorff = max(
        sp.spatial.distance.directed_hausdorff(tool_points_1, tool_points_2)[0],
        sp.spatial.distance.directed_hausdorff(tool_points_2, tool_points_1)[0],
    )
    return mse, hausdorff

In [ ]:
def video_path_tool_mask_video_path2df(video_path, tool_mask_video_path):
    with VideoFrames(
        video_path, step_ms=step_ms, color_conversion=cv2.COLOR_BGR2RGB, quiet=True
    ) as video_frames:
        image_frames = [
            rescale_to_height(f, rescale_target_height, 1)
            for _, _, f in tqdm(video_frames, leave=False)
        ]
    with VideoFrames(
        tool_mask_video_path,
        step_ms=step_ms,
        color_conversion=cv2.COLOR_BGR2RGB,
        quiet=True,
    ) as video_frames:
        tool_mask_frames = [
            si.transform.resize(
                f, image_frames[0].shape[:2], order=0, anti_aliasing=False
            )
            for _, _, f in tqdm(video_frames, leave=False)
        ]
    if len(image_frames) != len(tool_mask_frames):
        warnings.warn(
            f"For {video_path=}, {tool_mask_video_path=}: {len(image_frames)=} != {len(tool_mask_frames)=}"
        )
        return None
    frame_idxs = range(len(image_frames))
    video_df = []
    for idx_1, idx_2 in tqdm(list(combinations(frame_idxs, 2)), leave=False):
        mse, hausdorff = image_arrs_tool_mask_arrs2mse_hausdorff(
            image_frames[idx_1],
            image_frames[idx_2],
            tool_mask_frames[idx_1],
            tool_mask_frames[idx_2],
        )
        if not m.isfinite(mse) or not m.isfinite(hausdorff):
            continue
        video_df.append((idx_1, idx_2, mse, hausdorff))
    video_df = pd.DataFrame(video_df, columns=["idx_1", "idx_2", "MSE", "Hausdorff"])
    return video_df

In [ ]:
video_dfs = [
    video_path_tool_mask_video_path2df(video_path, tool_mask_video_path)
    for video_path, tool_mask_video_path in zip(
        tqdm(video_paths), tool_mask_video_paths
    )
]

In [ ]:
# From https://stackoverflow.com/a/40239615
def is_pareto_efficient(costs, return_mask=True):
    """
    Find the pareto-efficient points
    :param costs: An (n_points, n_costs) array
    :param return_mask: True to return a mask
    :return: An array of indices of pareto-efficient points.
        If return_mask is True, this will be an (n_points, ) boolean array
        Otherwise it will be a (n_efficient_points, ) integer array of indices.
    """
    is_efficient = np.arange(costs.shape[0])
    n_points = costs.shape[0]
    next_point_index = 0  # Next index in the is_efficient array to search for
    while next_point_index < len(costs):
        nondominated_point_mask = np.any(costs < costs[next_point_index], axis=1)
        nondominated_point_mask[next_point_index] = True
        is_efficient = is_efficient[nondominated_point_mask]  # Remove dominated points
        costs = costs[nondominated_point_mask]
        next_point_index = np.sum(nondominated_point_mask[:next_point_index]) + 1
    if return_mask:
        is_efficient_mask = np.zeros(n_points, dtype=bool)
        is_efficient_mask[is_efficient] = True
        return is_efficient_mask
    else:
        return is_efficient

In [ ]:
MSE_q = 0.5
Hausdorff_q = 0.5
min_selected_pairs_per_video = 50

In [ ]:
for video_df in video_dfs:
    video_df[["MSE", "Hausdorff"]] -= video_df[["MSE", "Hausdorff"]].mean()
    video_df[["MSE", "Hausdorff"]] /= video_df[["MSE", "Hausdorff"]].std()
    MSE_lt_q = video_df["MSE"] < video_df["MSE"].quantile(MSE_q)
    Hausdorff_gt_q = video_df["Hausdorff"] > video_df["Hausdorff"].quantile(Hausdorff_q)
    M_l_H_g_q = MSE_lt_q & Hausdorff_gt_q
    Pareto_eff = is_pareto_efficient(
        np.vstack((video_df["MSE"], -video_df["Hausdorff"])).T
    )
    selected = M_l_H_g_q & Pareto_eff
    if selected.sum() < min_selected_pairs_per_video:
        distance_to_selected = sp.spatial.distance_matrix(
            video_df[["MSE", "Hausdorff"]], video_df.loc[selected, ["MSE", "Hausdorff"]]
        ).min(axis=-1)
        distance_to_selected_argsorted = np.argsort(distance_to_selected)
        selected_idxs = distance_to_selected_argsorted[:min_selected_pairs_per_video]
        video_df["selected"] = False
        video_df.loc[selected_idxs, "selected"] = True
    else:
        video_df["selected"] = selected
    print(video_df["selected"].sum())

In [ ]:
for video_df in video_dfs:
    px.scatter(
        video_df,
        x="MSE",
        y="Hausdorff",
        hover_data=["idx_1", "idx_2"],
        color="selected",
    ).show()

In [ ]:
# !mkdir data/tool_bias_frame_pair_metadata

In [ ]:
# for video_path, video_df in zip(video_paths, video_dfs):
#     video_df.to_csv(join('data/tool_bias_frame_pair_metadata/', basename(video_path).removesuffix('.mov') + '.csv'), index=False)

In [ ]:
video_dfs = [
    pd.read_csv(
        join(
            "data/tool_bias_frame_pair_metadata/",
            basename(video_path).removesuffix(".mov") + ".csv",
        )
    )
    for video_path in video_paths
]

In [ ]:
for video_df in video_dfs:
    px.scatter(
        video_df,
        x="MSE",
        y="Hausdorff",
        hover_data=["idx_1", "idx_2"],
        color="selected",
    ).show()

In [ ]:
!mkdir data/tool_bias_frame_pair_metadata/frames
for video_path, video_df in zip(tqdm(video_paths), video_dfs):
    !mkdir data/tool_bias_frame_pair_metadata/frames/{basename(video_path).removesuffix('.mov')}
    selected_frame_idxs = set(video_df["idx_1"].unique()) | set(
        video_df["idx_2"].unique()
    )
    with VideoFrames(
        video_path, step_ms=step_ms, color_conversion=cv2.COLOR_BGR2RGB, quiet=True
    ) as video_frames:
        for i, (_, _, f) in enumerate(tqdm(video_frames, leave=False)):
            image_frame = rescale_to_height(f, rescale_target_height, 1)
            si.io.imsave(
                f'data/tool_bias_frame_pair_metadata/frames/{basename(video_path).removesuffix(".mov")}/{i}.png',
                si.util.img_as_ubyte(image_frame),
            )